# 1. Import Library 

In [ ]:
import os
import pandas as pd
import numpy as np
import pyodbc


# 2. Connection to SQL Server

In [ ]:
conn = pyodbc.connect( #CONNECTION
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-OQLFA92\\SQLEXPRESS;"
    "DATABASE=vti_dataset;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;")




df = pd.read_sql_query(  #Data Frame
'''                                        
SELECT *
FROM [vti_dataset].[dbo].[Final Project];                         
''', conn)

df


# 3. Preprocessing Data

In [ ]:
df.shape

# DataFrame have shape of (11552, 85)

In [ ]:
df.info()

# We observe missing data in data frame

In [ ]:
df.describe()

# Statistical summary of numerical columns

# CompTotal Salary: Mean =  8.5 USD , Median =  6.5 USD , std 9.6 USD

    # => We observe than mean > median , so the distribution is right skewed 

    # => We see high standard deviation , so the data is widely spreaded => High salary, some persons have outlier salary, STD spred 

# WorkWeekHrs: Mean =  41.0 Hrs , Median =  40.0 Hrs , std 8.0 Hrs

# CodeRevHrs: Mean =  3.5 Hrs , Median =  2.0 Hrs , std 4.5 Hrs

#...

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns
cat_cols = [c for c in cat_cols if c != "Respondent"]

output_path = "categorical_summary_all.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write("CATEGORICAL VARIABLE SUMMARY (Grouped by Respondent)\n")
    f.write("=" * 60 + "\n")

    for col in cat_cols:
        distinct_cnt = df[col].nunique(dropna=True)

        if distinct_cnt < 20:
            f.write(f"\n=== {col} (distinct={distinct_cnt}) ===\n")
            counts = (
                df.groupby(col)["Respondent"]
                  .nunique()
                  .sort_values(ascending=False)
            )

            for category, cnt in counts.items():
                f.write(f"{category}: {cnt}\n")
        else:
            f.write(f"\n--- SKIPPED {col} (distinct={distinct_cnt}) ---\n")

#Insights Cateogryrical Variables But it is hard explain here

In [ ]:
from openai import OpenAI

with open("categorical_summary_all.txt", "r", encoding="utf-8") as f:
    txt_content = f.read()

client = OpenAI(
    api_key="sk-"
)

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": f"""
Analyze the following categorical respondent summary.
Give 5 key insights, anomalies, and high-level observations.

TEXT:
{txt_content}
"""}
    ]
)

print(response.output_text)


In [ ]:
# Remove Outliers x Duplicates and Missing Values

import pandas as pd

def clean_final_df_no_outlier_removal(df):
    df = df.copy()
    impact = {}

    # --------------------
    # Initial rows
    # --------------------
    impact["initial_rows"] = len(df)

    # --------------------
    # 1️⃣ Remove duplicates
    # --------------------
    impact["duplicate_rows"] = int(df.duplicated().sum())
    df = df.drop_duplicates()

    # --------------------
    # 2️⃣ Fill missing values
    # --------------------
    num_cols = df.select_dtypes(include=["number"]).columns
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    impact["missing_filled"] = {}

    # Numeric → mean
    for col in num_cols:
        cnt = df[col].isna().sum()
        if cnt > 0:
            mean_val = df[col].mean()
            df[col].fillna(mean_val, inplace=True)
            impact["missing_filled"][col] = int(cnt)

    # Categorical → mode
    for col in cat_cols:
        cnt = df[col].isna().sum()
        if cnt > 0:
            mode_val = df[col].mode().iloc[0]
            df[col].fillna(mode_val, inplace=True)
            impact["missing_filled"][col] = int(cnt)

    # --------------------
    # 3️⃣ Cap outliers (IQR) — NO ROW REMOVAL
    # --------------------
    impact["outliers_capped"] = {}

    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        before_cap = ((df[col] < lower) | (df[col] > upper)).sum()
        df[col] = df[col].clip(lower, upper)

        if before_cap > 0:
            impact["outliers_capped"][col] = int(before_cap)

    # --------------------
    # Final rows
    # --------------------
    impact["final_rows"] = len(df)

    return df, impact


df_final, impact_report = clean_final_df_no_outlier_removal(df)

impact_report

# Final cleaned DataFrame and impact report: 11398

# We remove 154 duplicate rows and fill missing values without removing any rows.

# Outlier is not suitable to run (few columns are numerical).

# 4. Exploration Data Analytics (EDA)

In [ ]:
# Biz Needs:

    #--Các ngôn ngữ lập trình nào đang được yêu cầu nhiều nhất? #ngonngulaptrinh

    #--Các kỹ năng về cơ sở dữ liệu nào đang được yêu cầu nhiều nhất? #kynangcosodulieu

    #--Các Môi trường Phát triển Tích hợp (IDE) phổ biến là gì? #moitruongphattrientichhop

from openai import OpenAI

with open("schema.txt", "r", encoding="utf-8") as f:
    txt_content = f.read()

client = OpenAI(
    api_key="sk-"
)

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": f"""

    #--Các ngôn ngữ lập trình nào đang được yêu cầu nhiều nhất? #ngonngulaptrinh

    #--Các kỹ năng về cơ sở dữ liệu nào đang được yêu cầu nhiều nhất? #kynangcosodulieu

    #--Các Môi trường Phát triển Tích hợp (IDE) phổ biến là gì? #moitruongphattrientichhop

Analyze the following schema summary. Answers which columns are relevant to answer the above business needs from txt_content.

Return it ### Summary of Relevant Columns by Business Needs:

TEXT:
{txt_content}
"""}
    ]
)

print(response.output_text)

### Support us to read huge schema text and give relevant columns:


In [ ]:
#1. **Các ngôn ngữ lập trình nào đang được yêu cầu nhiều nhất? #ngonngulaptrinh**
# Relevant Columns: "LanguageWorkedWith", "LanguageDesireNextYear"

from collections import Counter

language_counter = Counter()

for langs in df["LanguageWorkedWith"].dropna():
    for lang in langs.split(";"):
        language_counter[lang.strip()] += 1

# Convert to DataFrame if needed
lang_count_df = (
    pd.DataFrame(language_counter.items(), columns=["Language", "Appear_Count"])
    .sort_values("Appear_Count", ascending=False)
)
lang_count_df.head(10)

# We observe that JavaScript, HTML/CSS, SQL are top 3 languages used by developers.

In [ ]:
#2. **Các kỹ năng về cơ sở dữ liệu nào đang được yêu cầu nhiều nhất? #kynangcosodulieu**

from collections import Counter

language_counter = Counter()

for langs in df["DatabaseWorkedWith"].dropna():
    for lang in langs.split(";"):
        language_counter[lang.strip()] += 1

# Convert to DataFrame if needed
lang_count_df = (
    pd.DataFrame(language_counter.items(), columns=["DatabaseWorkedWith", "Appear_Count"])
    .sort_values("Appear_Count", ascending=False)
)
lang_count_df.head(10)

## We observe that DataFrames, MySQL, and PostgreSQL are the top 3 databases among developers.

In [ ]:
#3. **Các Môi trường Phát triển Tích hợp (IDE) phổ biến là gì? #moitruongphattrientichhop**

from collections import Counter

language_counter = Counter()

for langs in df["DevEnviron"].dropna():
    for lang in langs.split(";"):
        language_counter[lang.strip()] += 1

# Convert to DataFrame if needed
lang_count_df = (
    pd.DataFrame(language_counter.items(), columns=["DevEnviron", "Appear_Count"])
    .sort_values("Appear_Count", ascending=False)
)
lang_count_df.head(10)

# We observe that Visual Studio Code, Visual Studio, and Jupyter Notebook are the top 3 popular IDEs among developers.

# 5. STAST Analyze

In [ ]:
#Hypothesis Chi-Squared Test: https://www.youtube.com/watch?v=HKDqlYSLt68

#Does the amount of time spent on code review (CodeRevHrs) depend on whether someone works remotely (WorkRemote)?

import pandas as pd
from scipy.stats import chi2_contingency

df1 = df[["CodeRevHrs", "WorkRemote"]].dropna()

ct = pd.crosstab(df1["CodeRevHrs"], df1["WorkRemote"])

chi2, p, dof, expected = chi2_contingency(ct)

print("Chi2:", chi2)
print("p-value:", p)

In [ ]:
#Does employment type affect job satisfaction? 
df2 = df[["Employment", "JobSat"]].dropna()

ct = pd.crosstab(df2["Employment"], df2["JobSat"])

chi2, p, dof, expected = chi2_contingency(ct)

print("Chi2:", chi2)
print("p-value:", p)

In [ ]:
#🔹 Education Level × Developer Type
df3 = df[["EdLevel", "DevType"]].dropna()
df3 = df3.assign(
    DevType=df3["DevType"].str.split(";")
).explode("DevType")

ct = pd.crosstab(df3["EdLevel"], df3["DevType"])

chi2, p, dof, expected = chi2_contingency(ct)

print("Chi2:", chi2)
print("p-value:", p)

In [ ]:
from itertools import combinations

results = []

for col1, col2 in combinations(cat_cols, 2):
    tmp = df[[col1, col2]].dropna()

    # Skip small samples
    if tmp.shape[0] < 30:
        continue

    ct = pd.crosstab(tmp[col1], tmp[col2])

    # Skip invalid contingency tables
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        continue

    try:
        chi2, p, dof, _ = chi2_contingency(ct)

        results.append({
            "Variable_1": col1,
            "Variable_2": col2,
            "Chi2": chi2,
            "p_value": p,
            "DoF": dof,
            "Sample_Size": ct.values.sum()
        })

    except ValueError:
        continue

chi2_results_df = (
    pd.DataFrame(results)
    .sort_values("Chi2", ascending=False)
    .head(10)
)

In [ ]:
chi2_results_df ### DATA MINING RESULTS

# 6. Save Data in SQL Server

In [ ]:
import pyodbc

# --- Connect ---
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-OQLFA92\\SQLEXPRESS;"
    "DATABASE=vti_dataset;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
cursor = conn.cursor()

def dtype_map(dt):
    if pd.api.types.is_integer_dtype(dt):
        return "INT"
    elif pd.api.types.is_float_dtype(dt):
        return "FLOAT"      # safe after cleaning
    elif pd.api.types.is_bool_dtype(dt):
        return "BIT"
    elif pd.api.types.is_datetime64_any_dtype(dt):
        return "DATETIME"
    else:
        return "VARCHAR(MAX)"   # IMPORTANT FIX


table_name = "Final_Project_EDA" #

# --- Drop + Create table ---
cursor.execute(f"IF OBJECT_ID('dbo.{table_name}', 'U') IS NOT NULL DROP TABLE dbo.{table_name}")
cols = ", ".join([f"[{c}] {dtype_map(df[c].dtype)}" for c in df.columns]) #
cursor.execute(f"CREATE TABLE dbo.{table_name} ({cols})")


# --- Insert rows ---
cursor.fast_executemany = True
cursor.executemany(f"INSERT INTO dbo.{table_name} VALUES ({','.join(['?']*len(df.columns))})", df.values.tolist()) #
conn.commit()

In [ ]:
table_name = "Final_Project_EDA" #

# --- Drop + Create table ---
cursor.execute(f"IF OBJECT_ID('dbo.{table_name}', 'U') IS NOT NULL DROP TABLE dbo.{table_name}")
cols = ", ".join([f"[{c}] {dtype_map(df[c].dtype)}" for c in df.columns]) #
cursor.execute(f"CREATE TABLE dbo.{table_name} ({cols})")


# --- Insert rows ---
cursor.fast_executemany = True
cursor.executemany(f"INSERT INTO dbo.{table_name} VALUES ({','.join(['?']*len(df.columns))})", df.values.tolist()) #
conn.commit()